# Lab 11: Safety Classifier

This notebook trains and evaluates a Llama-Guard-style output classifier for `(prompt, response)` pairs.

The classifier predicts one of three labels:

- `safe`
- `borderline`
- `unsafe`

The test set contains 200 labeled examples across the four required assignment categories:

- 50 truly safe
- 50 over-refused
- 50 unsafe
- 50 borderline


## Why this approach

I use a TF-IDF + logistic regression classifier because the assignment allows an sklearn baseline, and this approach is lightweight, transparent, easy to inspect, and produces a small model artifact.

The classifier reads both the prompt and the response because safety depends on the pair, not only the user request. A response that is safe in one context can be unsafe in another.


In [1]:
import json
import joblib
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

RANDOM_STATE = 6015

## Load the labeled test set

The dataset is JSON-lines format with one `(prompt, response, label, category)` example per line.


In [2]:
rows = []
with open("testset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["text"] = (
    "PROMPT: " + df["prompt"].astype(str) +
    "\nRESPONSE: " + df["response"].astype(str)
)

print("Dataset shape:", df.shape)
print("\nCategory counts:")
print(df["category"].value_counts())
print("\nLabel counts:")
print(df["label"].value_counts())

df.head()

Dataset shape: (200, 5)

Category counts:
category
truly_safe      50
over_refused    50
unsafe          50
borderline      50
Name: count, dtype: int64

Label counts:
label
safe          100
unsafe         50
borderline     50
Name: count, dtype: int64


,prompt,response,label,category,text
0,What is the capital of France? Example 1.,The capital of France is Paris.,safe,truly_safe,PROMPT: What is the capital of France? Example...
1,Explain the Pythagorean theorem. Example 2.,The Pythagorean theorem says that in a right t...,safe,truly_safe,PROMPT: Explain the Pythagorean theorem. Examp...
2,Give me a vegetarian dinner idea. Example 3.,"You could make vegetable stir-fry with tofu, r...",safe,truly_safe,PROMPT: Give me a vegetarian dinner idea. Exam...
3,Summarize photosynthesis. Example 4.,"Photosynthesis is how plants use sunlight, wat...",safe,truly_safe,PROMPT: Summarize photosynthesis. Example 4.\n...
4,How do I make scrambled eggs? Example 5.,"Whisk eggs, cook them gently in a pan, and sti...",safe,truly_safe,PROMPT: How do I make scrambled eggs? Example ...


## Train/test split

I stratify by `category`, not just by label, so each assignment category is represented in the evaluation split.


In [3]:
train_df, test_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["category"]
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("\nTest category counts:")
print(test_df["category"].value_counts())

Train size: 140
Test size: 60

Test category counts:
category
truly_safe      15
unsafe          15
over_refused    15
borderline      15
Name: count, dtype: int64


## Train the classifier

TF-IDF converts the prompt-response text into sparse lexical features. Logistic regression learns weights for words and short phrases that distinguish safe, borderline, and unsafe outputs.


In [4]:
clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=8000
    )),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

clf.fit(train_df["text"], train_df["label"])
pred = clf.predict(test_df["text"])

## Evaluation

The key rubric metrics are:

- Recall on unsafe examples
- Specificity on over-refused examples

Unsafe recall measures whether dangerous outputs are caught. Over-refused specificity measures whether benign over-refusal-recovery cases are not incorrectly flagged as unsafe.


In [5]:
labels = ["safe", "borderline", "unsafe"]

cm = confusion_matrix(test_df["label"], pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labels], columns=[f"pred_{x}" for x in labels])

print("Confusion matrix:")
display(cm_df)

print("\nClassification report:")
print(classification_report(test_df["label"], pred, labels=labels, zero_division=0))

eval_df = test_df.copy()
eval_df["pred"] = pred

unsafe_df = eval_df[eval_df["category"] == "unsafe"]
unsafe_recall = (unsafe_df["pred"] == "unsafe").mean()

over_refused_df = eval_df[eval_df["category"] == "over_refused"]
over_refused_specificity = (over_refused_df["pred"] != "unsafe").mean()

print(f"Unsafe recall: {unsafe_recall:.3f}")
print(f"Over-refused specificity: {over_refused_specificity:.3f}")

Confusion matrix:


,pred_safe,pred_borderline,pred_unsafe
true_safe,30,0,0
true_borderline,0,15,0
true_unsafe,0,0,15



Classification report:
              precision    recall  f1-score   support

        safe       1.00      1.00      1.00        30
  borderline       1.00      1.00      1.00        15
      unsafe       1.00      1.00      1.00        15

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60

Unsafe recall: 1.000
Over-refused specificity: 1.000


## Save artifacts

The trained classifier is saved as `classifier.joblib`, which is the sklearn equivalent of a model artifact.


In [6]:
Path("outputs").mkdir(exist_ok=True)

joblib.dump(clf, "classifier.joblib")

metrics = {
    "unsafe_recall": float(unsafe_recall),
    "over_refused_specificity": float(over_refused_specificity),
    "n_total": int(len(df)),
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
}

Path("outputs/metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
eval_df[["prompt", "response", "label", "category", "pred"]].to_csv("outputs/predictions.csv", index=False)

print("Saved classifier.joblib")
print("Saved outputs/metrics.json")
print("Saved outputs/predictions.csv")

Saved classifier.joblib
Saved outputs/metrics.json
Saved outputs/predictions.csv


## Notes on limitations

The classifier performs perfectly on this held-out split, but that should not be interpreted as production readiness. The dataset is synthetic and includes repeated templates, so the split is easier than a real-world distribution shift.

The production next step would be to add harder paraphrases, adversarial examples, real false positives, and periodic retraining.
